# Project 3 Evaluation (obs3 version)

⚠️ **IMPORTANT (Jupyter Kernel Selection)**:
Before running this notebook, please select the Python interpreter inside the **`.venv`** folder (located in the project root directory) as your Jupyter kernel. Please refer to `readme.md` for more details. All required libraries are pre-installed in this environment.

This notebook is designed specifically for **policy execution/evaluation** without training. It loads pre-trained policy arrays and neural network weights (from the `policy_obs3/` folder) and runs immediate evaluation trials, rendering gameplay videos.

### How to Run:
Set your notebook kernel to `.venv`, then run all cells sequentially. It will evaluate the following pre-trained models:
1. Tabular Q-Learning (Loads Q-table from `.npy`)
2. Tabular SARSA (Loads Q-table from `.npy`)
3. DQN (Grayscale & RGB) (Loads PyTorch weights from `.pth`)
4. DQN with Early Stopping (Grayscale & RGB) (Loads PyTorch weights from `.pth`)
5. DQN Hyperparameter Variants (Grayscale ES BS=256, Grayscale ES Decay=0.95) (Loads PyTorch weights from `.pth`)

## 1. Imports and Environment Setup

In [ ]:
import os
import random
import math
import datetime
import io
import imageio
from base64 import b64encode
from IPython.display import HTML, display
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from minigrid.wrappers import RGBImgObsWrapper
from minigrid.minigrid_env import MiniGridEnv, MissionSpace
from minigrid.core.grid import Grid
from minigrid.core.world_object import Goal, Wall

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
class SimpleEnv(MiniGridEnv):
    # Set size to 8 to secure internal 6x6 space
    def __init__(self, size=8, agent_start_pos=(1,1), agent_start_dir=0, max_steps: int | None = None, agent_view_size=3, **kwargs):
        self.agent_start_pos = agent_start_pos
        self.agent_start_dir = agent_start_dir
        mission_space = MissionSpace(mission_func=self._gen_mission)
        super().__init__(mission_space=mission_space, grid_size=size, max_steps=100, agent_view_size=agent_view_size, **kwargs)

    @staticmethod
    def _gen_mission():
        return "grand mission"

    def _gen_grid(self, width, height):
        self.grid = Grid(width, height)
        self.grid.wall_rect(0, 0, width, height)

        # [Custom Complex Maze] Create crossroads and dead ends

        # 1. Top part: Lead agent to a dead end to deceive it
        self.grid.set(2, 2, Wall())
        self.grid.set(3, 2, Wall())
        self.grid.set(4, 1, Wall())
        self.grid.set(4, 2, Wall())
        self.grid.set(6, 2, Wall())

        # 2. Middle part: Create walls and bifurcated paths
        self.grid.set(1, 4, Wall())
        self.grid.set(3, 4, Wall())
        self.grid.set(4, 4, Wall())
        self.grid.set(6, 3, Wall())

        # 3. Bottom part: Obstacle in front of final destination
        self.grid.set(3, 5, Wall())
        self.grid.set(5, 6, Wall())

        # Set Goal position at (6, 6)
        self.grid.set(width - 2, height - 2, Goal())

        # Set agent start position and direction
        if self.agent_start_pos is not None:
            self.agent_pos = self.agent_start_pos
            self.agent_dir = self.agent_start_dir
        else:
            self.place_agent()

        self.mission = "grand mission"

### Preview the Environment Layout

In [ ]:
env = SimpleEnv(render_mode="rgb_array")
env.reset()
env.unwrapped.highlight = False

image_data = env.render()
plt.imshow(image_data)
plt.axis("off")
plt.show()

## 2. Helper Functions for Evaluation and Preprocessing

In [ ]:
# 1. Video generation and display function
def display_video(frames, fps=10, filename=None):
    if isinstance(fps, str):
        filename = fps
        fps = 10
    os.makedirs("video_obs3", exist_ok=True)
    if filename:
        save_path = os.path.join("video_obs3", filename)
        imageio.mimsave(save_path, frames, format="mp4", fps=fps)
        print(f"Video saved to {save_path}")
    video_buffer = io.BytesIO()
    imageio.mimsave(video_buffer, frames, format="mp4", fps=fps)
    video_buffer.seek(0)
    video_base64 = b64encode(video_buffer.read()).decode()

    video_html = f"""
    <video width="640" height="480" controls autoplay>
    <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
    </video>
    """
    display(HTML(video_html))

# 2. State preprocessing for DQN input
def pre_state(obs, use_grayscale=True):
    state = obs['image'] / 255.0
    state = torch.tensor(state, dtype=torch.float32).permute(2, 0, 1)
    if use_grayscale:
        to_grayscale = transforms.Grayscale(num_output_channels=1)
        state = to_grayscale(state)
    return state

## 3. Tabular Q-Learning & SARSA Evaluation

In [ ]:
def test_tabular_and_display(Q_values, title="Tabular Model"):
    env = SimpleEnv(render_mode="rgb_array")
    obs, info = env.reset(seed=42)

    # Initialize state list
    state = np.array([env.agent_pos[0]-1, env.agent_pos[1]-1, obs["direction"]])
    done = False
    frames = []
    frames.append(env.render())

    i = 0
    while not done:
        x, y, d = state[0], state[1], state[2]
        action = np.argmax(Q_values[x, y, d])

        # Take action in the environment
        next_obs, reward, terminated, truncated, info = env.step(action)
        state = np.array([env.agent_pos[0]-1, env.agent_pos[1]-1, next_obs["direction"]])
        done = terminated or truncated

        frames.append(env.render())
        i += 1

        # Prevent the agent from falling into an infinite loop if it fails to find the path
        if i > 100:
            print(f"{title}: Shortest path search failed (Prevented infinite loop)")
            break

    env.close()
    print(f"{title}: Finished at the {i}-th iteration")
    return frames

In [ ]:
print("--- Loading and Evaluating Tabular Q-Learning ---")
learned_Q_q_learning = np.load("policy_obs3/learned_Q_q_learning.npy")
q_frames = test_tabular_and_display(learned_Q_q_learning, title="Q-Learning")
display_video(q_frames, filename="q_learning_play.mp4")

In [ ]:
print("--- Loading and Evaluating Tabular SARSA ---")
learned_Q_sarsa = np.load("policy_obs3/learned_Q_sarsa.npy")
sarsa_frames = test_tabular_and_display(learned_Q_sarsa, title="SARSA")
display_video(sarsa_frames, filename="sarsa_play.mp4")

## 4. Deep Q-Network (DQN) Setup & Helper Functions

In [ ]:
class Model(nn.Module):
    def __init__(self, input_shape, num_actions):
        super(Model, self).__init__()
        in_channels = input_shape[0]
        
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=5, stride=2, padding=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1)
        
        self.fc1 = nn.Linear(32 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, num_actions)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

def load_model(model, run_name, filename=None):
    if filename is None:
        filename = f"{run_name}_save.pth"
    if not filename.startswith("policy_obs3/") and not filename.startswith("policy_obs3\\"):
        path = os.path.join("policy_obs3", filename)
    else:
        path = filename
    model.load_state_dict(torch.load(path, map_location=device))
    print(f"Model loaded from {path}")
    return model

def test_dqn(model, use_grayscale=True):
    env = SimpleEnv(render_mode="rgb_array")
    env = RGBImgObsWrapper(env)
    obs, info = env.reset(seed=42)
    state = pre_state(obs, use_grayscale=use_grayscale)
    done = False
    frames = [env.render()]
    
    steps = 0
    while not done and steps < 100:
        state_tensor = state.unsqueeze(0).to(device)
        with torch.no_grad():
            action = model(state_tensor).argmax(dim=1).item()
        next_obs, reward, terminated, truncated, info = env.step(action)
        state = pre_state(next_obs, use_grayscale=use_grayscale)
        done = terminated or truncated
        frames.append(env.render())
        steps += 1
        
    env.close()
    print(f"Finished in {steps} steps")
    return frames

## 5. Standard DQN Evaluation

In [ ]:
print("--- Testing Grayscale Model ---")
model_grayscale = Model((1, 64, 64), 3).to(device)
model_grayscale = load_model(model_grayscale, "DQN_Grayscale")
grayscale_frames = test_dqn(model_grayscale, use_grayscale=True)
display_video(grayscale_frames, filename="dqn_grayscale_play.mp4")

In [ ]:
print("--- Testing RGB Model ---")
model_rgb = Model((3, 64, 64), 3).to(device)
model_rgb = load_model(model_rgb, "DQN_RGB")
rgb_frames = test_dqn(model_rgb, use_grayscale=False)
display_video(rgb_frames, filename="dqn_rgb_play.mp4")

## 6. DQN with Early Stopping Evaluation

In [ ]:
print("--- Testing Grayscale ES Model ---")
model_grayscale_es = Model((1, 64, 64), 3).to(device)
model_grayscale_es = load_model(model_grayscale_es, "DQN_Grayscale_ES")
grayscale_es_frames = test_dqn(model_grayscale_es, use_grayscale=True)
display_video(grayscale_es_frames, filename="dqn_grayscale_es_play.mp4")

In [ ]:
print("--- Testing RGB ES Model ---")
model_rgb_es = Model((3, 64, 64), 3).to(device)
model_rgb_es = load_model(model_rgb_es, "DQN_RGB_ES")
rgb_es_frames = test_dqn(model_rgb_es, use_grayscale=False)
display_video(rgb_es_frames, filename="dqn_rgb_es_play.mp4")

## 7. DQN Hyperparameter Comparison Evaluation

In [ ]:
print("--- Testing Grayscale ES (BS=256) Model ---")
model_grayscale_es_bs256 = Model((1, 64, 64), 3).to(device)
model_grayscale_es_bs256 = load_model(model_grayscale_es_bs256, "DQN_Grayscale_ES_BS256")
grayscale_es_bs256_frames = test_dqn(model_grayscale_es_bs256, use_grayscale=True)
display_video(grayscale_es_bs256_frames, filename="dqn_grayscale_es_bs256_play.mp4")

In [ ]:
print("--- Testing Grayscale ES (Decay=0.95) Model ---")
model_grayscale_es_ed095 = Model((1, 64, 64), 3).to(device)
model_grayscale_es_ed095 = load_model(model_grayscale_es_ed095, "DQN_Grayscale_ES_ED095")
grayscale_es_ed095_frames = test_dqn(model_grayscale_es_ed095, use_grayscale=True)
display_video(grayscale_es_ed095_frames, filename="dqn_grayscale_es_ed095_play.mp4")